In [ ]:
# 1. Data Preparation & Pre-Processing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv('WI_and_features.csv', sep=';')
X = df.drop(['id', 'wealth_index'], axis=1)
y = df['wealth_index']
ids = df['id']

In [ ]:
### 1.1 Multicollinearity Reduction & Train-Test Split

In [ ]:
# (Opsional) Menghilangkan fitur yang multikolinear
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones_like(corr_matrix), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]
X_reduced = X.drop(to_drop, axis=1)

# Gunakan X jika tidak mereduksi, atau X_reduced jika mereduksi
# Pada script asli menggunakan X penuh untuk split awal:
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, ids, test_size=0.3, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 2. Permutation Feature Importances for Dimension Reduction

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

# Train initial model for PFI
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

feature_names = X_train.columns if hasattr(X_train, 'columns') else np.arange(X_train.shape[1])

def PFI(model, X_test, y_test, n=50, feature_names=feature_names):
    # Compute permutation importance (use scaled test data!)
    result = permutation_importance(
        model, 
        X_test, 
        y_test, 
        n_repeats=10, 
        random_state=42
    )
    
    importances = result.importances_mean
    top_n = n
    top_indices = np.argsort(importances)[-top_n:] 
    top_importances = importances[top_indices] 
    
    # Plot
    plt.figure(figsize=(12, 8))
    plt.title(f"Top {top_n} Feature Importances")
    plt.bar(range(top_n), top_importances, align="center")
    plt.xticks(range(top_n), feature_names[top_indices], rotation=90)
    plt.xlim([-1, top_n])
    plt.ylabel("Permutation Importance Score")
    plt.tight_layout()
    plt.show()
    
    return top_indices, top_importances

# Jalankan PFI untuk mengambil top 25 fitur
top_indices, top_importances = PFI(model, X_test_scaled, y_test, 25)

# Select top N features for reduced dataset
selected_features = feature_names[top_indices]
X_train_reduced = X_train_scaled[:, top_indices] 
X_test_reduced = X_test_scaled[:, top_indices]

In [ ]:
# 3. Model Training

In [ ]:
from tpot import TPOTRegressor

# Initialize default model
default_model = RandomForestRegressor(random_state=42)

# Define hyperparameter grid for TPOT config
params_grid = {
    'n_estimators': [50, 100, 200], 
    'max_depth': [10, 20, 30], 
    'min_samples_split': [10, 15, 20, 30], 
    'min_samples_leaf': [4, 6, 8, 10, 15, 20], 
    'max_features': ['sqrt']
}

# K-Fold Cross-Validation
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Initialize TPOT model
tpot_model = TPOTRegressor(
    generations=10, 
    population_size=50, 
    offspring_size=12,
    verbosity=2, 
    early_stop=12,
    config_dict={'sklearn.ensemble.RandomForestRegressor': params_grid},
    cv=kfold, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1
)

# Fitting both models
default_model.fit(X_train_reduced, y_train)
tpot_model.fit(X_train_reduced, y_train)

In [ ]:
# 4. Model Evaluation & Visualization

In [ ]:
def evaluate_model(model, X, y):
    pred = model.predict(X)
    return {
        'RMSE': np.sqrt(mean_squared_error(y, pred)),
        'MAE': mean_absolute_error(y, pred),
        'R²': r2_score(y, pred),
        'rRMSE': np.sqrt(mean_squared_error(y, pred)) / np.mean(y) * 100
    }

default_scores = evaluate_model(default_model, X_test_reduced, y_test)
optimized_scores = evaluate_model(tpot_model, X_test_reduced, y_test)

# Create comparison dataframe
comparison = pd.DataFrame({
    'Default Model': default_scores,
    'Optimized Model': optimized_scores,
    'Improvement': {
        'RMSE': default_scores['RMSE'] - optimized_scores['RMSE'],
        'MAE': default_scores['MAE'] - optimized_scores['MAE'],
        'R²': default_scores['R²'] - optimized_scores['R²']
    }
})

print("\n=== Performance Comparison ===")
print(comparison)

# Optional secondary PFI on optimized model
f_names = selected_features
top_indices, top_importances = PFI(tpot_model, X_test_reduced, y_test, 10, selected_features)

# Actual vs Predicted comparison plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

ax1.scatter(y_test, default_model.predict(X_test_reduced), alpha=0.6)
ax1.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
ax1.set_title('Default Model: Actual vs Predicted')
ax1.set_xlabel('Actual')
ax1.set_ylabel('Predicted')

ax2.scatter(y_test, tpot_model.predict(X_test_reduced), alpha=0.6)
ax2.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2)
ax2.set_title('Optimized Model: Actual vs Predicted')
ax2.set_xlabel('Actual')
ax2.set_ylabel('Predicted')

plt.tight_layout()
plt.show()

# Metric comparison bar chart
plt.figure(figsize=(10, 6))
metrics = ['RMSE', 'MAE', 'R²']
x = np.arange(len(metrics))
width = 0.35
plt.bar(x - width/2, [default_scores[m] for m in metrics], width, label='Default')
plt.bar(x + width/2, [optimized_scores[m] for m in metrics], width, label='Optimized')
plt.xticks(x, metrics)
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.legend()
plt.show()

# Check Overfitting
default_train_pred = default_model.predict(X_train_reduced)
default_train_rmse = np.sqrt(mean_squared_error(y_train, default_train_pred))
default_train_r2 = r2_score(y_train, default_train_pred)

tpot_train_pred = tpot_model.predict(X_train_reduced)
tpot_train_rmse = np.sqrt(mean_squared_error(y_train, tpot_train_pred))
tpot_train_r2 = r2_score(y_train, tpot_train_pred)

print('Default Model Overfitting Performance:')
print(f"Train RMSE: {default_train_rmse:.4f} vs Test RMSE: {default_scores['RMSE']:.4f}")
print(f"Train R2: {default_train_r2:.4f} vs Test R2: {default_scores['R²']:.4f}")
print('Score Ratio: ' + str(default_train_rmse / default_scores['RMSE']))

print('\nOptimized Model Overfitting Performance:')
print(f"Train RMSE: {tpot_train_rmse:.4f} vs Test RMSE: {optimized_scores['RMSE']:.4f}")
print(f"Train R2: {tpot_train_r2:.4f} vs Test R2: {optimized_scores['R²']:.4f}")
print('Score Ratio: ' + str(tpot_train_rmse / optimized_scores['RMSE']))